# Lecture 04 · Teach a Robot with PPO 🤖  — PPO from scratch (continuous control)

**RL in Production · Vizuara AI Labs**

In Lecture 3 you climbed the variance ladder — REINFORCE → +baseline → Actor-Critic —
on the discrete Archer. Lecture 4 takes the next step: the robot's actuators are
**continuous** (a torque, a thruster), and we want updates that are **big enough to
learn fast** but **small enough not to fall off a cliff**. That tension is the whole
story of TRPO → **PPO**.

You will implement **Proximal Policy Optimization from scratch in PyTorch** (no
Stable-Baselines3) and teach a robot two continuous-control tasks:

> **Pendulum-v1** (swing a torque-limited arm upright) → **LunarLanderContinuous-v3** (land a lander with two analog thrusters)

The two ideas that make PPO work are exactly the two you will build:

1. **The clipped surrogate** — let the policy ratio $r=\pi/\pi_{\text{old}}$ move,
   but **clip** it to $[1-\varepsilon,\,1+\varepsilon]$ so one batch can't shove the
   policy too far. This is the trust-region idea from TRPO, made cheap.
2. **Sample reuse** — because clipping keeps each step honest, we can safely take
   **K gradient epochs** over the *same* batch instead of throwing it away.

You will also run the lecture's **ablation**: turn the clip off, turn the reuse off,
and watch each one break. That is the experiment that proves *why* PPO is built the
way it is.

**Cells marked ✅ PROVIDED are complete — run them. Cells marked 📝 TODO are yours
to finish** (each has the equation from the slide right next to it). Everything
downstream calls your code, so fill the TODOs top to bottom.

Runs on a free CPU Colab — Pendulum trains in ~1–2 minutes.

## Part 0 · Setup (✅ provided)

If you are on Colab, the first cell installs the Box2D physics backend (needed for
the LunarLander stretch goal). On a plain CPU runtime this is all you need.

In [ ]:
# ✅ PROVIDED — Colab install (Box2D powers LunarLanderContinuous). Safe to re-run.
# On Colab this prints a lot; that's fine. Skip if you already have gymnasium[box2d].
!pip install -q "gymnasium[box2d]" 2>/dev/null || pip install -q "gymnasium[box2d]"
print("setup cell done")

In [ ]:
# ✅ PROVIDED — imports + seeding
import numpy as np
import torch
import torch.nn as nn
import gymnasium as gym
import matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

# Hyperparameters (match the lecture reference — don't change these for the main task)
CLIP_EPS  = 0.2     # PPO clip range  [1-eps, 1+eps]
EPOCHS    = 10      # K gradient epochs over each batch (the "reuse")
MIN_STEPS = 2048    # transitions collected per iteration before an update
MB_SIZE   = 64      # minibatch size inside the K-epoch loop
LR        = 3e-4    # Adam learning rate (both actor and critic)
print("imports OK · torch", torch.__version__, "· gymnasium", gym.__version__)

## Part 1 · The networks (✅ provided)

A robot's action is a **real-valued vector** (Pendulum: one torque in $[-2, 2]$;
LunarLander: two thruster commands in $[-1, 1]$). So the policy can't output logits
over a handful of choices like the Archer did — instead it outputs a **Gaussian**:

- a **mean** action $\mu(s)$ from the network, and
- a **state-independent log-std** parameter (one learnable number per action dim).

We sample $a \sim \mathcal{N}(\mu(s), \sigma)$ to explore, and at eval time we just
take the mean (greedy). The **critic** $V(s)$ is the same baseline idea as L3 — it
predicts the return so we can form an advantage.

In [ ]:
# ✅ PROVIDED — the Gaussian policy (actor) and the value network (critic)
class GaussianPolicy(nn.Module):
    """Actor: state -> mean action; plus a state-independent log-std parameter."""
    def __init__(self, obs_dim, act_dim, h=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(obs_dim, h), nn.Tanh(),
                                  nn.Linear(h, h), nn.Tanh())
        self.mean = nn.Linear(h, act_dim)
        self.log_std = nn.Parameter(torch.zeros(act_dim) - 0.5)   # std ~ 0.6 to start

    def dist(self, obs):
        m = self.mean(self.body(obs))
        return torch.distributions.Normal(m, self.log_std.exp())  # a Gaussian per action dim


class Value(nn.Module):
    """Critic: state -> V(s), a single scalar."""
    def __init__(self, obs_dim, h=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(obs_dim, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(), nn.Linear(h, 1))

    def forward(self, x):
        return self.net(x).squeeze(-1)

## Part 2 · Rollout collection (✅ provided)

PPO is **on-policy**: each iteration we collect a fresh batch with the *current*
policy. We run **complete episodes** until we have at least `MIN_STEPS` transitions,
storing for every step:

- the state `S` and the action `A` we took,
- **`logp_old`** — the log-probability of that action *under the policy that took it*.

That `logp_old` is the linchpin of PPO. We freeze it now (`torch.no_grad`), and in
the update we compare it against the log-prob under the *new* policy to form the
ratio $r=\pi/\pi_{\text{old}}$. Notice the action is clipped to the robot's physical
limits before stepping the environment.

In [ ]:
# ✅ PROVIDED — collect complete episodes with the current policy
def collect(env, policy, gamma, min_steps=MIN_STEPS):
    """Run COMPLETE episodes until >= min_steps transitions are collected.
    Returns flat arrays: states, actions, OLD log-probs, MC returns G, mean ep return."""
    S, A, LOGP, RET, ep_rets = [], [], [], [], []
    lo, hi = env.action_space.low, env.action_space.high
    steps = 0
    while steps < min_steps:
        s, _ = env.reset()
        ep_s, ep_a, ep_lp, ep_r = [], [], [], []
        done = False
        while not done:
            st = torch.as_tensor(s, dtype=torch.float32)
            with torch.no_grad():
                dist = policy.dist(st)
                a = dist.sample()
                lp = dist.log_prob(a).sum(-1)        # sum log-probs over action dims
            a_np = np.clip(a.numpy(), lo, hi)        # keep within the robot's limits
            s2, r, term, trunc, _ = env.step(a_np)
            ep_s.append(s); ep_a.append(a.numpy()); ep_lp.append(float(lp)); ep_r.append(r)
            s = s2; done = term or trunc; steps += 1
        S += ep_s; A += ep_a; LOGP += ep_lp
        RET += discounted_returns(ep_r, gamma)       # uses YOUR TODO 1 below
        ep_rets.append(sum(ep_r))
    return (np.array(S, np.float32), np.array(A, np.float32),
            np.array(LOGP, np.float32), np.array(RET, np.float32), float(np.mean(ep_rets)))

### 📝 TODO 1 — discounted return-to-go $G_t$

Same recursion as L3, computed backwards over one episode:

$$G_t = r_t + \gamma\,r_{t+1} + \gamma^2 r_{t+2} + \dots = r_t + \gamma\,G_{t+1}.$$

This is the **Monte-Carlo return** — the actual discounted reward the robot earned
from step $t$ onward. The critic will learn to predict it, and the advantage is
built from it. (No TD bootstrapping, no GAE — just the real return, like the deck.)

In [ ]:
# 📝 TODO 1 — complete the accumulation line
def discounted_returns(rewards, gamma):
    """G_t = r_t + gamma * G_{t+1}, computed backwards (reward-to-go)."""
    G, out = 0.0, []
    for r in reversed(rewards):
        G = ...                # 📝 TODO: one step of the recursion   (hint: r + gamma * G)
        out.append(G)
    return list(reversed(out))

## Part 3 · The advantage $A = G - V$ (📝 TODO)

The actor should push up actions that did **better than expected** and push down
ones that did worse. "Expected" is the critic's prediction $V(s)$, so the
**advantage** is the return minus that baseline:

$$A_t = G_t - V(s_t).$$

This is the *exact* baseline from L3 — a Monte-Carlo return minus a learned value,
**no GAE, no $\lambda$, no TD-error**. We compute it **once per batch** (the critic
is frozen for this step with `no_grad`), then **standardize** it (subtract mean,
divide by std) so the update size doesn't swing with the reward scale.

In [ ]:
# 📝 TODO 2 — complete the advantage, then it gets standardized for you
def compute_advantage(value, S, RET):
    """A = G - V(s), computed once per batch, then normalized to mean 0 / std 1."""
    with torch.no_grad():
        adv = ...          # 📝 TODO: advantage = return minus critic baseline
                           #          hint:  RET - value(S)
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)   # ✅ standardize (provided)
    return adv

## Part 4 · THE CORE — the PPO clipped update (📝 TODO)

This is the heart of the lecture. For each minibatch we:

**1. Recompute the log-prob under the *current* policy** and form the **ratio**

$$r = \frac{\pi(a\mid s)}{\pi_{\text{old}}(a\mid s)} = \exp\big(\log\pi - \log\pi_{\text{old}}\big).$$

At the very first epoch $r=1$ (the policy hasn't moved yet); as we take gradient
steps it drifts away from 1.

**2. Form the clipped surrogate.** We want to *maximize* $r\cdot A$, but clip $r$ so
one batch can't move the policy too far. As a **loss to minimize**:

$$L^{\text{CLIP}} = -\,\mathbb{E}\Big[\min\big(r\,A,\;\;\text{clip}(r,\,1-\varepsilon,\,1+\varepsilon)\,A\big)\Big].$$

The `min` is the trust region: when an action looks good ($A>0$) the clip caps how
much we reward increasing its probability; when it looks bad ($A<0$) it caps how
hard we push it down. Either way the policy can only move so far per batch.

**3. The value loss** trains the critic to predict the return: $L^V = (V(s) - G)^2$.

**4. The K-epoch reuse loop** — because clipping keeps each step safe, we sweep the
same batch `K = EPOCHS` times in random minibatches (when `use_reuse=False`, K=1, so
each batch is used once — the ablation).

The two switches `use_clip` and `use_reuse` are wired in so you can run the ablation
in Part 6 *with the same function*. Fill the three marked lines.

In [ ]:
# 📝 TODO 3 (THE CORE) — complete the ratio and the clipped surrogate
def ppo_update(policy, value, popt, vopt, batch, *, clip_eps=CLIP_EPS, epochs=EPOCHS,
               mb_size=MB_SIZE, use_clip=True, use_reuse=True):
    S, A, LOGP_old, RET, _ = batch
    S = torch.as_tensor(S); A = torch.as_tensor(A)
    LOGP_old = torch.as_tensor(LOGP_old); RET = torch.as_tensor(RET)

    adv = compute_advantage(value, S, RET)        # your TODO 2 (computed once, normalized)

    n = S.shape[0]
    K = epochs if use_reuse else 1                 # <-- the REUSE ablation: K epochs vs 1
    for _ in range(K):
        for idx in torch.randperm(n).split(mb_size):
            # --- actor: clipped surrogate ---
            dist = policy.dist(S[idx])
            logp = dist.log_prob(A[idx]).sum(-1)               # log pi(a|s) under CURRENT policy
            ratio = ...        # 📝 TODO: r = pi / pi_old      (hint: (logp - LOGP_old[idx]).exp())
            if use_clip:                                       # <-- the CLIP ablation
                unclipped = ratio * adv[idx]
                clipped   = ...    # 📝 TODO: clip the ratio to [1-eps, 1+eps], then * adv[idx]
                                   #          hint: torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv[idx]
                actor_loss = -torch.min(unclipped, clipped).mean()   # the L^CLIP objective
            else:
                actor_loss = -(ratio * adv[idx]).mean()        # no-clip = plain surrogate
            popt.zero_grad(); actor_loss.backward(); popt.step()

            # --- critic: regress V(s) onto the return G ---
            value_loss = ((value(S[idx]) - RET[idx]) ** 2).mean()    # ✅ provided
            vopt.zero_grad(); value_loss.backward(); vopt.step()

## Part 5 · Train + evaluate (✅ provided)

The train loop is the standard on-policy cycle: **collect a batch → PPO update →
repeat**, logging the mean episode return each iteration. `greedy_eval` runs the
policy deterministically (action = the Gaussian's mean) to measure final skill.

In [ ]:
# ✅ PROVIDED — the on-policy train loop (collect -> update -> repeat)
def train(env_id, iters=60, min_steps=MIN_STEPS, gamma=0.95, lr=LR,
          use_clip=True, use_reuse=True, epochs=EPOCHS, log_every=10, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    env = gym.make(env_id); env.reset(seed=seed)
    od = env.observation_space.shape[0]; ad = env.action_space.shape[0]
    pol = GaussianPolicy(od, ad); val = Value(od)
    popt = torch.optim.Adam(pol.parameters(), lr)
    vopt = torch.optim.Adam(val.parameters(), lr)
    curve = []
    for it in range(iters):
        batch = collect(env, pol, gamma, min_steps)
        curve.append(batch[4])                                  # mean episode return this iter
        ppo_update(pol, val, popt, vopt, batch, epochs=epochs,
                   use_clip=use_clip, use_reuse=use_reuse)
        if (it + 1) % log_every == 0:
            print(f"  {env_id}  iter {it+1:3d}  mean episode return {np.mean(curve[-5:]):8.1f}")
    env.close()
    return pol, val, curve

In [ ]:
# ✅ PROVIDED — greedy (deterministic) evaluation: action = the Gaussian mean
def greedy_eval(env_id, policy, n=10, seed=123):
    env = gym.make(env_id); lo, hi = env.action_space.low, env.action_space.high
    tot = 0.0
    for i in range(n):
        s, _ = env.reset(seed=seed + i); done = False
        while not done:
            with torch.no_grad():
                a = policy.dist(torch.as_tensor(s, dtype=torch.float32)).mean   # greedy = mean
            s, r, term, trunc, _ = env.step(np.clip(a.numpy(), lo, hi)); tot += r; done = term or trunc
    env.close()
    return tot / n

### Train PPO on Pendulum-v1 (✅ provided)

Swing a torque-limited arm upright and hold it. Reward is near 0 when balanced and
very negative when hanging — so you should watch the curve climb **from around
−1100 toward roughly −700 to −500** over 60 iterations (about 1–2 minutes on CPU).
`gamma = 0.95` here (short horizon).

In [ ]:
# ✅ PROVIDED — train, plot the learning curve, and report greedy skill
pol, val, curve = train("Pendulum-v1", iters=60, gamma=0.95)
print(f"\nPendulum:  start {np.mean(curve[:3]):.1f}  ->  end {np.mean(curve[-3:]):.1f}"
      f"   |   greedy {greedy_eval('Pendulum-v1', pol):.1f}")

plt.figure(figsize=(8, 4))
plt.plot(curve, alpha=.4, label='per-iteration')
plt.plot(np.convolve(curve, np.ones(5)/5, 'valid'), lw=2, label='smoothed (5)')
plt.xlabel('PPO iteration'); plt.ylabel('mean episode return')
plt.title('PPO on Pendulum-v1 — the robot learns to swing up'); plt.legend(); plt.grid(alpha=.3); plt.show()

## Part 6 · Ablation — why clip? why reuse? (📝 run + explain)

The two switches you wired into `ppo_update` let you turn each PPO ingredient off
**without changing anything else**, so any difference you see is caused by that one
change. Run all three on Pendulum (40 iters each, ~3–4 min total) and watch:

- **full PPO** — clip on, K=10 reuse on,
- **NO clip** — `use_clip=False`: the surrogate is unbounded, so a single batch can
  shove the policy off a cliff,
- **NO reuse** — `use_reuse=False` (K=1): each batch is used once, so it learns far
  more slowly per sample.

You should see **full PPO clearly best**, NO-reuse slower, and NO-clip the most
unstable / worst. Then answer the questions in the next cell.

In [ ]:
# 📝 PROVIDED harness — just RUN this (it calls your ppo_update with each switch)
ablation = {}
for label, kw in [("full PPO", {}),
                  ("NO clip",  {"use_clip": False}),
                  ("NO reuse", {"use_reuse": False})]:
    print(f"--- {label} ---")
    _, _, cc = train("Pendulum-v1", iters=40, gamma=0.95, log_every=40, **kw)
    ablation[label] = cc

plt.figure(figsize=(8, 4))
for label, cc in ablation.items():
    plt.plot(np.convolve(cc, np.ones(5)/5, 'valid'), lw=2, label=label)
plt.xlabel('PPO iteration'); plt.ylabel('mean episode return (smoothed)')
plt.title('Ablation on Pendulum — clip and reuse each matter'); plt.legend(); plt.grid(alpha=.3); plt.show()
for label, cc in ablation.items():
    print(f"  {label:10s} start {np.mean(cc[:3]):8.1f}  ->  end {np.mean(cc[-3:]):8.1f}")

### 📝 Your ablation write-up

Answer in the cell below (a few sentences each), reading off **your** curves:

1. **Clip.** How does turning the clip off change the curve? Tie it to the surrogate
   objective: with no clip, what stops a single over-large ratio $r$ from moving the
   policy too far in one batch? (This is the TRPO trust-region motivation.)
2. **Reuse.** With `use_reuse=False` (K=1) each batch is used once. How does learning
   speed *per iteration* change, and why is it **safe** to reuse a batch K times in
   full PPO but riskier without the clip?
3. **Together.** Rank the three runs by final return. Which single ingredient hurt
   most when removed, and does that match the lecture's claim that PPO = (trust
   region via clipping) + (sample efficiency via reuse)?

In [ ]:
# 📝 Your ablation answers here (as comments, or convert this to a markdown cell)


## Part 7 · Stretch goal — LunarLanderContinuous-v3 🚀

Same PPO code, a harder robot: a lander with **two analog thrusters** (main + side).
Land it gently between the flags. This task has a longer horizon, so use
**`gamma = 0.99`** and more iterations. On CPU this takes longer (~5–10 min) — a good
batch run. You're aiming to climb **from around −200 up past +200** (a solved lander).

Nothing new to implement — if your TODOs are right, this just works. 🎉

In [ ]:
# ✅ PROVIDED — stretch: train PPO on LunarLanderContinuous-v3 (gamma 0.99)
pol_ll, val_ll, curve_ll = train("LunarLanderContinuous-v3", iters=150, gamma=0.99, log_every=10)
print(f"\nLunarLander:  start {np.mean(curve_ll[:3]):.1f}  ->  end {np.mean(curve_ll[-3:]):.1f}"
      f"   |   greedy {greedy_eval('LunarLanderContinuous-v3', pol_ll):.1f}")

plt.figure(figsize=(8, 4))
plt.plot(curve_ll, alpha=.4, label='per-iteration')
plt.plot(np.convolve(curve_ll, np.ones(5)/5, 'valid'), lw=2, label='smoothed (5)')
plt.axhline(200, ls='--', c='g', label='solved (+200)')
plt.xlabel('PPO iteration'); plt.ylabel('mean episode return')
plt.title('PPO on LunarLanderContinuous-v3 — learning to land'); plt.legend(); plt.grid(alpha=.3); plt.show()

## Part 8 · Deliverable & where to go next

**Deliverable:** this notebook with all 📝 TODOs filled (TODO 1 the return, TODO 2
the advantage, **TODO 3 the PPO clipped update**), the Pendulum learning curve and
the ablation plot rendered, and your Part 6 write-up. Submit the `.ipynb` (or a
Colab share link). The LunarLander stretch is optional but recommended.

**Concept companions (revisit alongside this notebook):**
- *From TRPO to PPO* — why a trust region, and how clipping approximates it cheaply.
- *The surrogate objective* — the importance-sampling ratio $r$ and what `min(...)`
  actually does to the gradient when $A>0$ vs $A<0$.

**Play with the PPO Visualizer** — drag the advantage and the ratio and watch the
clipped surrogate flatten exactly where your `torch.clamp` kicks in. Seeing the
hinge move is the fastest way to build intuition for what you just coded.

Nicely done — you've built the algorithm that trains most real-world RL robots. 🤖